# Lab 7 — FGSM Adversarial Attacks

**Student:** Hitesh Gupta | **Roll No.:** IC-2k23-39

Fast Gradient Sign Method (FGSM) crafts adversarial examples by
adding a small perturbation in the direction of the loss gradient:

`x_adv = x + ε · sign(∇_x J(θ, x, y))`

We evaluate ResNet-18's robustness on ImageNet samples.

In [ ]:
import torch
import torch.nn.functional as F
from torchvision.models import resnet18, ResNet18_Weights
import torchvision.transforms as T
import numpy as np
import matplotlib.pyplot as plt
import os

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
os.makedirs('outputs', exist_ok=True)

## 1. Load Model & Data

In [ ]:
weights = ResNet18_Weights.DEFAULT
model = resnet18(weights=weights).to(device)
model.eval()
preprocess = weights.transforms()
categories = weights.meta['categories']

# Use a synthetic batch if no real images available
sample_paths = ['inputs/sample1.jpg', 'inputs/sample2.jpg',
                '../datasets/sample1.jpg']
found = [p for p in sample_paths if os.path.exists(p)]

if found:
    from PIL import Image
    images = []
    for p in found[:4]:
        img = Image.open(p).convert('RGB')
        images.append(preprocess(img))
    x = torch.stack(images).to(device)
else:
    print('No sample images — generating random batch.')
    x = torch.randn(4, 3, 224, 224, device=device)

## 2. FGSM Attack Implementation

In [ ]:
def fgsm_attack(model, x, y, epsilon):
    """Generate adversarial examples via FGSM."""
    x_adv = x.clone().detach().requires_grad_(True)
    output = model(x_adv)
    loss = F.cross_entropy(output, y)
    model.zero_grad()
    loss.backward()
    grad = x_adv.grad.data
    x_adv = x_adv + epsilon * grad.sign()
    x_adv = torch.clamp(x_adv, 0, 1)
    return x_adv.detach()

def evaluate(model, x, y):
    with torch.no_grad():
        out = model(x)
        pred = out.argmax(dim=1)
        acc = (pred == y).float().mean().item()
    return acc, pred

## 3. Attack at Multiple Epsilon Values

In [ ]:
# Get true labels from clean predictions
with torch.no_grad():
    clean_out = model(x)
    y_true = clean_out.argmax(dim=1)

epsilons = [0.0, 0.01, 0.03, 0.05, 0.1, 0.2]
results = []

for eps in epsilons:
    if eps == 0.0:
        x_adv = x
    else:
        x_adv = fgsm_attack(model, x, y_true, eps)
    acc, pred = evaluate(model, x_adv, y_true)
    results.append({'epsilon': eps, 'accuracy': acc})
    print(f'epsilon={eps:.2f}  accuracy={acc:.3f}')

res_df = __import__('pandas').DataFrame(results)
res_df

In [ ]:
# Accuracy vs epsilon curve
plt.figure(figsize=(8, 4))
plt.plot(res_df['epsilon'], res_df['accuracy'], 'o-', color='crimson')
plt.xlabel('Epsilon (perturbation magnitude)')
plt.ylabel('Model Accuracy')
plt.title('FGSM Attack — Accuracy vs Epsilon')
plt.grid(True, alpha=0.3)
plt.savefig('outputs/fgsm_accuracy.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Visualise original vs adversarial for one sample
eps_demo = 0.1
x_adv_demo = fgsm_attack(model, x[:1], y_true[:1], eps_demo)

def tensor_to_img(t):
    # Undo normalisation approximately for display
    mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1).to(device)
    std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1).to(device)
    img = t * std + mean
    img = img.clamp(0, 1).squeeze().permute(1, 2, 0).cpu().numpy()
    return img

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(tensor_to_img(x[:1]))
axes[0].set_title('Original')
axes[0].axis('off')
axes[1].imshow(tensor_to_img(x_adv_demo))
axes[1].set_title(f'Adversarial (eps={eps_demo})')
axes[1].axis('off')
# Perturbation amplified
perturb = (x_adv_demo - x[:1]).abs().mean(dim=1, keepdim=True)
perturb = perturb.squeeze().cpu().numpy()
axes[2].imshow(perturb, cmap='hot')
axes[2].set_title('Perturbation magnitude')
axes[2].axis('off')
plt.tight_layout()
plt.savefig('outputs/fgsm_examples.png', dpi=300, bbox_inches='tight')
plt.show()

## Analysis

- FGSM is a **white-box** attack requiring gradient access.
- Even small `ε` (0.01–0.05) can flip predictions while being
  imperceptible to humans.
- Accuracy drops sharply as `ε` increases, demonstrating model
  vulnerability to adversarial perturbations.
- Defences include adversarial training, input preprocessing, and
  gradient masking.